In [1]:
import pandas as pd
import os
from tqdm import tqdm
import monai
import json
tqdm.pandas()

/projects/ceib/python_enviroments/monai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_mids = pd.read_csv('/home/jaalzate/Tartaglia/Prostate_Tartaglia/Tartaglia_variables/variablesPT4_deps_ready_final_new_MIDS.tsv', sep='\t')

df_mids['label_XNAT_session'] = df_mids['label_XNAT_session'].fillna(0)
df_mids['label_XNAT_session'] = df_mids['label_XNAT_session'].astype(int).astype(str).str.zfill(6)
df_mids['label_MIDS_session'] = df_mids['label_XNAT_session'].apply(lambda x: 'ses-'+x)

df_mids['label_XNAT_subject'] = df_mids['label_XNAT_subject'].fillna(0)
df_mids['label_XNAT_subject'] = df_mids['label_XNAT_subject'].astype(int).astype(str).str.zfill(6)
df_mids['label_MIDS_subject'] = df_mids['label_XNAT_subject'].apply(lambda x: 'sub-'+x)

In [3]:
df_mids

,project,dep,label_XNAT_subject,label_XNAT_session,ID_XNAT_subject,ID_XNAT_session,ED,AF,TB,TR,...,VP,PIR,csPC,F_nacimiento,F_RM,F_TACTO_RECTAL,F_PSA,F_csPC,label_MIDS_session,label_MIDS_subject
0,p0042021,7.0,000001,000001,XNAT3_S36738,XNAT3_E91781,77.0,NaN,1.0,NaN,...,48.472756,5.0,1.0,1934-10-02,2012-07-04,NaN,-136.0,-115.0,ses-000001,sub-000001
1,p0042021,2.0,000002,000002,XNAT3_S36739,XNAT3_E91782,57.0,NaN,NaN,NaN,...,26.282880,NaN,0.0,1958-08-10,2015-10-18,NaN,NaN,-51.0,ses-000002,sub-000002
2,p0042021,7.0,000003,000003,XNAT3_S36740,XNAT3_E91783,69.0,NaN,2.0,NaN,...,NaN,NaN,1.0,1945-01-02,2014-03-07,NaN,-42.0,-18.0,ses-000003,sub-000003
3,p0042021,7.0,000004,000004,XNAT3_S36741,XNAT3_E91784,52.0,NaN,2.0,NaN,...,NaN,NaN,1.0,1964-08-13,2017-03-24,NaN,-31.0,-63.0,ses-000004,sub-000004
4,p0042021,7.0,000005,000005,XNAT3_S36742,XNAT3_E91785,71.0,NaN,NaN,NaN,...,NaN,NaN,NaN,1944-02-16,2015-06-13,NaN,NaN,NaN,ses-000005,sub-000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10304,p0052021,17.0,005390,005809,XNAT3_S42195,XNAT3_E97620,73.0,NaN,1.0,NaN,...,NaN,NaN,1.0,1939-06-23,2013-05-27,NaN,-21.0,55.0,ses-005809,sub-005390
10305,p0052021,5.0,005391,005810,XNAT3_S42196,XNAT3_E97621,62.0,NaN,NaN,NaN,...,83.865600,NaN,0.0,1952-01-11,2014-05-15,NaN,-168.0,-52.0,ses-005810,sub-005391
10306,p0052021,5.0,005392,005811,XNAT3_S42197,XNAT3_E97622,71.0,NaN,NaN,NaN,...,NaN,NaN,0.0,1941-08-31,2012-10-24,NaN,-200.0,66.0,ses-005811,sub-005392
10307,p0052021,4.0,005393,005812,XNAT3_S42198,XNAT3_E97623,54.0,NaN,NaN,NaN,...,62.287680,3.0,0.0,1957-12-12,2011-12-13,NaN,NaN,476.0,ses-005812,sub-005393


In [13]:
df_mids['ID_XNAT_session'].nunique()

10309

In [5]:
df_mids[df_mids.project=='p0052021'].csPC.value_counts()

csPC
1.0    4328
0.0    3807
0.5     511
Name: count, dtype: int64

In [6]:
df_reports = pd.read_csv('/home/jaalzate/Files/reports_complete.csv')

In [7]:
df_reports

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,project,dep,ID_XNAT_subject,ID_XNAT_session,ED,AF,TB,...,PIR,csPC,F_nacimiento,F_RM,F_TACTO_RECTAL,F_PSA,F_csPC,VP_text,MRI_Report,reports_count
0,0,0,0,p0042021,7.0,XNAT3_S36738,XNAT3_E91781,77.0,NaN,1.0,...,[5],1.0,1934-10-02,2012-07-04,NaN,-136.0,-115.0,"48,1 x 38,3 x 50,6 mm,",INFORMACION CLINICA:\nCáncer de próstata G 3+4...,1
1,1,1,1,p0042021,2.0,XNAT3_S36739,XNAT3_E91782,57.0,NaN,NaN,...,NaN,0.0,1958-08-10,2015-10-18,NaN,NaN,-51.0,"medidas de 54 x 26 x 36 mm,",Exploración realizada:\nR M de próstata multip...,1
2,2,2,2,p0042021,7.0,XNAT3_S36740,XNAT3_E91783,69.0,NaN,2.0,...,NaN,1.0,1945-01-02,2014-03-07,NaN,-42.0,-18.0,NaN,JUICIO JUICIO Prostatectomía en Junio/2017. Pe...,1
3,3,3,3,p0042021,7.0,XNAT3_S36741,XNAT3_E91784,52.0,NaN,2.0,...,NaN,1.0,1964-08-13,2017-03-24,NaN,-31.0,-63.0,NaN,JUICIO JUICIO Tras prostatectomía R1 en apex. ...,1
4,4,4,4,p0042021,7.0,XNAT3_S36742,XNAT3_E91785,71.0,NaN,NaN,...,NaN,NaN,1944-02-16,2015-06-13,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10304,10304,10304,10304,p0052021,17.0,XNAT3_S42195,XNAT3_E97620,73.0,NaN,1.0,...,NaN,1.0,1939-06-23,2013-05-27,NaN,-21.0,55.0,NaN,MOTIVO MOTIVO MOTIVO Prostatectomía radical co...,1
10305,10305,10305,10305,p0052021,5.0,XNAT3_S42196,XNAT3_E97621,62.0,NaN,NaN,...,NaN,0.0,1952-01-11,2014-05-15,NaN,-168.0,-52.0,"64 x 56 x 45 mm,","TECNICA: Se practican planos axiales, sagitale...",1
10306,10306,10306,10306,p0052021,5.0,XNAT3_S42197,XNAT3_E97622,71.0,NaN,NaN,...,NaN,0.0,1941-08-31,2012-10-24,NaN,-200.0,66.0,NaN,"TECNICA: Se practican planos axiales, sagitale...",1
10307,10307,10307,10307,p0052021,4.0,XNAT3_S42198,XNAT3_E97623,54.0,NaN,NaN,...,[3],0.0,1957-12-12,2011-12-13,NaN,NaN,476.0,Próstata de 62 x 46 x 42 mm de diámetro transv...,Paciente con elevación de PSA de forma persist...,1


In [14]:
df_reports['ID_XNAT_session'].nunique()

10309

In [8]:
df_mids.columns

Index(['project', 'dep', 'label_XNAT_subject', 'label_XNAT_session',
       'ID_XNAT_subject', 'ID_XNAT_session', 'ED', 'AF', 'TB', 'TR', 'PSA',
       'VP', 'PIR', 'csPC', 'F_nacimiento', 'F_RM', 'F_TACTO_RECTAL', 'F_PSA',
       'F_csPC', 'label_MIDS_session', 'label_MIDS_subject'],
      dtype='object')

In [20]:
df_mids=df_mids[['ID_XNAT_subject', 'ID_XNAT_session','label_MIDS_session', 'label_MIDS_subject']]

In [21]:
df_reports.columns

Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'project', 'dep',
       'ID_XNAT_subject', 'ID_XNAT_session', 'ED', 'AF', 'TB', 'TR', 'PSA',
       'VP', 'PIR', 'csPC', 'F_nacimiento', 'F_RM', 'F_TACTO_RECTAL', 'F_PSA',
       'F_csPC', 'VP_text', 'MRI_Report', 'reports_count'],
      dtype='object')

In [22]:
df_merged=pd.merge(df_mids,df_reports,left_on='ID_XNAT_session',right_on='ID_XNAT_session')
df_merged

,ID_XNAT_subject_x,ID_XNAT_session,label_MIDS_session,label_MIDS_subject,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,project,dep,ID_XNAT_subject_y,...,PIR,csPC,F_nacimiento,F_RM,F_TACTO_RECTAL,F_PSA,F_csPC,VP_text,MRI_Report,reports_count
0,XNAT3_S36738,XNAT3_E91781,ses-000001,sub-000001,0,0,0,p0042021,7.0,XNAT3_S36738,...,[5],1.0,1934-10-02,2012-07-04,NaN,-136.0,-115.0,"48,1 x 38,3 x 50,6 mm,",INFORMACION CLINICA:\nCáncer de próstata G 3+4...,1
1,XNAT3_S36739,XNAT3_E91782,ses-000002,sub-000002,1,1,1,p0042021,2.0,XNAT3_S36739,...,NaN,0.0,1958-08-10,2015-10-18,NaN,NaN,-51.0,"medidas de 54 x 26 x 36 mm,",Exploración realizada:\nR M de próstata multip...,1
2,XNAT3_S36740,XNAT3_E91783,ses-000003,sub-000003,2,2,2,p0042021,7.0,XNAT3_S36740,...,NaN,1.0,1945-01-02,2014-03-07,NaN,-42.0,-18.0,NaN,JUICIO JUICIO Prostatectomía en Junio/2017. Pe...,1
3,XNAT3_S36741,XNAT3_E91784,ses-000004,sub-000004,3,3,3,p0042021,7.0,XNAT3_S36741,...,NaN,1.0,1964-08-13,2017-03-24,NaN,-31.0,-63.0,NaN,JUICIO JUICIO Tras prostatectomía R1 en apex. ...,1
4,XNAT3_S36742,XNAT3_E91785,ses-000005,sub-000005,4,4,4,p0042021,7.0,XNAT3_S36742,...,NaN,NaN,1944-02-16,2015-06-13,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10304,XNAT3_S42195,XNAT3_E97620,ses-005809,sub-005390,10304,10304,10304,p0052021,17.0,XNAT3_S42195,...,NaN,1.0,1939-06-23,2013-05-27,NaN,-21.0,55.0,NaN,MOTIVO MOTIVO MOTIVO Prostatectomía radical co...,1
10305,XNAT3_S42196,XNAT3_E97621,ses-005810,sub-005391,10305,10305,10305,p0052021,5.0,XNAT3_S42196,...,NaN,0.0,1952-01-11,2014-05-15,NaN,-168.0,-52.0,"64 x 56 x 45 mm,","TECNICA: Se practican planos axiales, sagitale...",1
10306,XNAT3_S42197,XNAT3_E97622,ses-005811,sub-005392,10306,10306,10306,p0052021,5.0,XNAT3_S42197,...,NaN,0.0,1941-08-31,2012-10-24,NaN,-200.0,66.0,NaN,"TECNICA: Se practican planos axiales, sagitale...",1
10307,XNAT3_S42198,XNAT3_E97623,ses-005812,sub-005393,10307,10307,10307,p0052021,4.0,XNAT3_S42198,...,[3],0.0,1957-12-12,2011-12-13,NaN,NaN,476.0,Próstata de 62 x 46 x 42 mm de diámetro transv...,Paciente con elevación de PSA de forma persist...,1


In [24]:
df_merged= df_merged.drop(columns=['ID_XNAT_subject_y','Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0','ID_XNAT_session','ID_XNAT_subject_x'])
df_merged

,label_MIDS_session,label_MIDS_subject,project,dep,ED,AF,TB,TR,PSA,VP,PIR,csPC,F_nacimiento,F_RM,F_TACTO_RECTAL,F_PSA,F_csPC,VP_text,MRI_Report,reports_count
0,ses-000001,sub-000001,p0042021,7.0,77.0,NaN,1.0,NaN,5.040000000000000,48.472756,[5],1.0,1934-10-02,2012-07-04,NaN,-136.0,-115.0,"48,1 x 38,3 x 50,6 mm,",INFORMACION CLINICA:\nCáncer de próstata G 3+4...,1
1,ses-000002,sub-000002,p0042021,2.0,57.0,NaN,NaN,NaN,NaN,26.282880,NaN,0.0,1958-08-10,2015-10-18,NaN,NaN,-51.0,"medidas de 54 x 26 x 36 mm,",Exploración realizada:\nR M de próstata multip...,1
2,ses-000003,sub-000003,p0042021,7.0,69.0,NaN,2.0,NaN,2.320000000000000,NaN,NaN,1.0,1945-01-02,2014-03-07,NaN,-42.0,-18.0,NaN,JUICIO JUICIO Prostatectomía en Junio/2017. Pe...,1
3,ses-000004,sub-000004,p0042021,7.0,52.0,NaN,2.0,NaN,0.09800000000000000,NaN,NaN,1.0,1964-08-13,2017-03-24,NaN,-31.0,-63.0,NaN,JUICIO JUICIO Tras prostatectomía R1 en apex. ...,1
4,ses-000005,sub-000005,p0042021,7.0,71.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1944-02-16,2015-06-13,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10304,ses-005809,sub-005390,p0052021,17.0,73.0,NaN,1.0,NaN,0.3600000000000000,NaN,NaN,1.0,1939-06-23,2013-05-27,NaN,-21.0,55.0,NaN,MOTIVO MOTIVO MOTIVO Prostatectomía radical co...,1
10305,ses-005810,sub-005391,p0052021,5.0,62.0,NaN,NaN,NaN,7.960000000000000,83.865600,NaN,0.0,1952-01-11,2014-05-15,NaN,-168.0,-52.0,"64 x 56 x 45 mm,","TECNICA: Se practican planos axiales, sagitale...",1
10306,ses-005811,sub-005392,p0052021,5.0,71.0,NaN,NaN,NaN,6.290000000000000,NaN,NaN,0.0,1941-08-31,2012-10-24,NaN,-200.0,66.0,NaN,"TECNICA: Se practican planos axiales, sagitale...",1
10307,ses-005812,sub-005393,p0052021,4.0,54.0,NaN,NaN,NaN,NaN,62.287680,[3],0.0,1957-12-12,2011-12-13,NaN,NaN,476.0,Próstata de 62 x 46 x 42 mm de diámetro transv...,Paciente con elevación de PSA de forma persist...,1


In [25]:
df_merged.iloc[0]

label_MIDS_session                                           ses-000001
label_MIDS_subject                                           sub-000001
project                                                        p0042021
dep                                                                 7.0
ED                                                                 77.0
AF                                                                  NaN
TB                                                                  1.0
TR                                                                  NaN
PSA                                                   5.040000000000000
VP                                                            48.472756
PIR                                                                 [5]
csPC                                                                1.0
F_nacimiento                                                 1934-10-02
F_RM                                                         201

In [26]:
df_merged.to_csv('/home/jaalzate/Files/reports_complete_MIDS.csv',index=False)